#### Import the Library

In [18]:
# NOTE: json = read JSON
# NOTE: Path = safer file paths than raw strings
# NOTE: pandas = convert to table + export CSV

import json
from pathlib import Path
import pandas as pd

#### File path + existence check

In [19]:
# NOTE: Put the file in the same folder as this notebook
# or change the path here.

input_path = Path("hard_events.json")
input_path

WindowsPath('hard_events.json')

In [20]:
# NOTE: always check the file exists before reading, saves time.

print("Exists ?", input_path.exists())
print("Absolute Path: ", input_path.resolve())

Exists ? True
Absolute Path:  C:\Users\pdinh\Python\Python_Full_Course\Python Practice\JSON-Python\json-practice-hard\hard_events.json


#### Read JSON only AFTER validation

In [21]:
# ------------------------------------------------------------
# STEP 3: Read the JSON file
# ------------------------------------------------------------

# NOTE:
# - open() reads the file as text
# - json.load() converts text -) Python obects (dict,list)
# - We DO NOT use pandas yet because pandas expects tabular logic

with input_path.open("r", encoding="utf-8") as f:
    raw = json.load(f)

# NOTE:
# At this moment:
# raw is NOT a DataFrame
# raw is a python dictionary (nested structure)
# NOTE: for json.loads is for text, json.load is for file

#### Inspect structure 

In [24]:
# ------------------------------------------------------------
# STEP 4: Inspect JSON structure
# ------------------------------------------------------------

# NOTE: check the keys from json
print("Top level keys: ", raw.keys())

# NOTE: check the meta key, normally they are for apis
print("Meta :", raw.get("meta"))

# NOTE: we work with data key, which is a list of orders
tran = raw.get("data")
print("Type of data: ",type(tran))
print("Number of orders: ", len(tran))
print("First order: ", tran[0])

Top level keys:  dict_keys(['meta', 'data'])
Meta : {'source': 'demo_api', 'pulled_at': '2026-02-05T00:00:00Z'}
Type of data:  <class 'list'>
Number of orders:  2
First order:  {'event_id': 'ev_9001', 'event_time': '2026-02-04T20:00:00Z', 'event_type': 'purchase', 'user': {'user_id': 301, 'traits': {'vip': True, 'signup_date': '2025-11-01'}, 'addresses': [{'type': 'home', 'city': 'Sydney', 'state': 'NSW'}, {'type': 'work', 'city': 'Sydney', 'state': 'NSW'}]}, 'payload': {'cart': {'items': [{'sku': 'X9', 'qty': 1, 'unit_price': 99.99}], 'coupon': 'WELCOME10'}, 'device': {'os': 'iOS', 'app_version': '5.1.0'}}}


In [25]:
tran[0]

{'event_id': 'ev_9001',
 'event_time': '2026-02-04T20:00:00Z',
 'event_type': 'purchase',
 'user': {'user_id': 301,
  'traits': {'vip': True, 'signup_date': '2025-11-01'},
  'addresses': [{'type': 'home', 'city': 'Sydney', 'state': 'NSW'},
   {'type': 'work', 'city': 'Sydney', 'state': 'NSW'}]},
 'payload': {'cart': {'items': [{'sku': 'X9', 'qty': 1, 'unit_price': 99.99}],
   'coupon': 'WELCOME10'},
  'device': {'os': 'iOS', 'app_version': '5.1.0'}}}

#### Use pandas to normalize Json, put all of them to a tabular format


In [44]:
events = pd.json_normalize(tran,sep=".")
events

,event_id,event_time,event_type,user.user_id,user.traits.vip,user.traits.signup_date,user.addresses,payload.cart.items,payload.cart.coupon,payload.device.os,payload.device.app_version,payload.page.url,payload.page.referrer
0,ev_9001,2026-02-04T20:00:00Z,purchase,301,True,2025-11-01,"[{'type': 'home', 'city': 'Sydney', 'state': '...","[{'sku': 'X9', 'qty': 1, 'unit_price': 99.99}]",WELCOME10,iOS,5.1.0,NaN,NaN
1,ev_9002,2026-02-04T20:05:00Z,click,302,False,NaN,NaN,NaN,NaN,Android,5.0.3,/home,NaN


##### Build fact_events (core event table)

In [38]:
fact_events = pd.json_normalize(tran)

# Keep key columns + flatten only what we need at event level
fact_events = fact_events[[
    "event_id",
    "event_time",
    "event_type",
    "user.user_id"
]].rename(columns={"user.user_id": "user_id"})

# Parse datetime (UTC)
fact_events["event_time"] = pd.to_datetime(fact_events["event_time"], utc=True, errors="coerce")

# Standardize user_id -> Int64 (nullable integer)
fact_events["user_id"] = pd.to_numeric(fact_events["user_id"], errors="coerce").astype("Int64")

fact_events


,event_id,event_time,event_type,user_id
0,ev_9001,2026-02-04 20:00:00+00:00,purchase,301
1,ev_9002,2026-02-04 20:05:00+00:00,click,302


#### Buid dim_users

In [40]:
dim_users = pd.json_normalize(tran, sep=".")

dim_users = dim_users[[
    "user.user_id",
    "user.traits.vip",
    "user.traits.signup_date"
]].rename(columns={
    "user.user_id": "user_id",
    "user.traits.vip": "vip",
    "user.traits.signup_date": "signup_date",
})

dim_users["user_id"] = pd.to_numeric(dim_users["user_id"], errors="coerce").astype("Int64")
dim_users["vip"] = dim_users["vip"].astype("boolean")
dim_users["signup_date"] = pd.to_datetime(dim_users["signup_date"], errors="coerce").dt.date

# De-duplicate (one row per user)
dim_users = dim_users.drop_duplicates(subset=["user_id"]).reset_index(drop=True)

dim_users


,user_id,vip,signup_date
0,301,True,2025-11-01
1,302,False,NaT


##### Build dim_addresses

In [45]:
rows = []
for ev in tran:
    user = ev.get("user", {})
    user_id = pd.to_numeric(user.get("user_id"), errors="coerce")
    for addr in user.get("addresses", []) or []:
        rows.append({
            "user_id": user_id,
            "address_type": addr.get("type"),
            "city": addr.get("city"),
            "state": addr.get("state")
        })

dim_addresses = pd.DataFrame(rows)

dim_addresses["user_id"] = dim_addresses["user_id"].astype("Int64")
dim_addresses = dim_addresses.drop_duplicates().reset_index(drop=True)

dim_addresses


,user_id,address_type,city,state
0,301,home,Sydney,NSW
1,301,work,Sydney,NSW


##### Build dim_device

In [48]:
dim_device = pd.json_normalize(tran, sep=".")

dim_device = dim_device[[
    "event_id",
    "payload.device.os",
    "payload.device.app_version"
]].rename(columns={
    "payload.device.os": "os",
    "payload.device.app_version": "app_version"
})

dim_device

,event_id,os,app_version
0,ev_9001,iOS,5.1.0
1,ev_9002,Android,5.0.3


#### buid dim_page

In [50]:
dim_page = pd.json_normalize(tran, sep=".")

dim_page = dim_page[[
    "event_id",
    "payload.page.url",
    "payload.page.referrer"
]].rename(columns={
    "payload.page.url": "url",
    "payload.page.referrer": "referrer"
})

# Keep rows where url exists (click events)
dim_page = dim_page[dim_page["url"].notna()].reset_index(drop=True)

dim_page


,event_id,url,referrer
0,ev_9002,/home,NaN


#### Build fact_cart_items

In [51]:
rows = []
for ev in tran:
    event_id = ev.get("event_id")
    event_type = ev.get("event_type")
    if event_type != "purchase":
        continue

    cart = (ev.get("payload") or {}).get("cart") or {}
    coupon = cart.get("coupon")

    for item in cart.get("items", []) or []:
        qty = pd.to_numeric(item.get("qty"), errors="coerce")
        unit_price = pd.to_numeric(item.get("unit_price"), errors="coerce")

        rows.append({
            "event_id": event_id,
            "sku": item.get("sku"),
            "qty": qty,
            "unit_price": unit_price,
            "line_total": qty * unit_price if pd.notna(qty) and pd.notna(unit_price) else None,
            "coupon": coupon
        })

fact_cart_items = pd.DataFrame(rows)
fact_cart_items

,event_id,sku,qty,unit_price,line_total,coupon
0,ev_9001,X9,1,99.99,99.99,WELCOME10
